# XBRL US API - Research facts in SEC reports by label 

### This notebook uses two looping queries: the first returns `concept` details for _keyword strings_ found in the human-readable labels of SEC reports, and; the second iterates the `concept` and discoverable taxonomy set (`dts`) values from the label search to return the most recent de-duplicated fact details for specified reporting periods and fiscal years.

**Authenticate for access token** - [get a free XBRL US account and request provisioning for the XBRL API](https://xbrl.us/access-token), then click the run button below to execute the cell's code. Enter your XBRL US Web account email, account password, Client ID, and secret as noted, pressing the Enter key on the keyboard after each entry. 

This script loops to collect all data from the Public Filings Database. XBRL US limits records returned improve efficiency and **non-Members might not be able to return all data for a query**. Join XBRL US for comprehensive access - https://xbrl.us/join.

In [ ]:
# @title
import os, re, sys, json
import requests
import pandas as pd
from IPython.display import display, HTML
import numpy as np
import getpass
from datetime import datetime
import urllib
from urllib.parse import urlencode


class tokenInfoClass:
    access_token = None
    refresh_token = None
    email = None
    username = None
    client_id = None
    client_secret = None
    url = 'https://api.xbrl.us/oauth2/token'
    headers = {"Content-Type": "application/x-www-form-urlencoded"}

def refresh(info):
    refresh_auth = {
                'client_id': info.client_id,
				'client_secret' : info.client_secret,
				'grant_type' : 'refresh_token',
				'platform' : 'ipynb',
				'refresh_token' : info.refresh_token
                }
    refreshres = requests.post(info.url, data=refresh_auth, headers=info.headers)
    refresh_json = refreshres.json()
    info.access_token = refresh_json['access_token']
    info.refresh_token = refresh_json['refresh_token']
    print('Your access token (%s) is refreshed for 60 minutes. If it expires again, run this cell to generate a new token and continue to use the query cells below.' % (info.access_token))
    return info

tokenInfo = tokenInfoClass()

tokenInfo.email = input('Enter your XBRL US Web account email: ')
tokenInfo.password = getpass.getpass(prompt='Password: ')
tokenInfo.client_id = getpass.getpass(prompt='Client ID: ')
tokenInfo.client_secret = getpass.getpass(prompt='Secret: ')

body_auth = {'username' : tokenInfo.email,
            'client_id': tokenInfo.client_id,
            'client_secret' : tokenInfo.client_secret,
            'password' : tokenInfo.password,
            'grant_type' : 'password',
            'platform' : 'ipynb' }

#print(body_auth)

payload = urlencode(body_auth)
res = requests.request("POST", tokenInfo.url, data=payload, headers=tokenInfo.headers)
auth_json = res.json()

if 'error' in auth_json:
    print('\n\nThere was a problem generating the access token: %s.  Run the first cell again and enter the credentials.' % (auth_json['error_description']))
else:
    tokenInfo.access_token = auth_json['access_token']
    tokenInfo.refresh_token = auth_json['refresh_token']
    print ('\n\nYour access token expires in 60 minutes. After it expires, it should be regenerated automatically.  If not, run the cell rerun the first query cell. \naccess token: ' + tokenInfo.access_token + ' refresh token: ' + tokenInfo.refresh_token + '\n\nFor now, skip ahead to the next section to define query parameters.')

#print(vars(tokenInfo))

# Define search filters and fields to return

The section below defines parameters for the `label` endpoint to evaluate `keyword strings` that appear as human-readable labels used in reports. Click in the code cell then click the run button to save the parameters, then run the next cell to query and return attributes for each concept.

In [ ]:
### Define the parameters for the filter and fields to be returned

# Define endpoint (common values: fact, entity, report, cube, label, concept, relationship - see https://xbrlus/github.io/xbrl-api for additional endpoint options)

endpoint = 'label'

Keyword_List = [
    #'government assistance',
    #'government grant',
    #'government incentive',
    'paycheck protect',
    'ppp ',
    #'tax incentive'
                ]

fields = [
         'label.text',
         'concept.local-name',
         'concept.id',
         'concept.namespace',
         'dts.id.sort(DESC)',
         'label.role-short',
         ]

# Set unique rows as True of False (True drops any duplicate rows)
unique = True

# Limit the number of rows displayed by the notebook (does not impact the data frame)
rows_to_display = 6 # Set as '' to display all rows in the notebook

params = {
     'concept.is-abstract': 'FALSE',
     'fields': ','.join(fields)
     }

print('\n\nClick the run button below to execute this query.\n\n')

In [ ]:
# @title
### Execute the query with loop for all results
### THIS SECTION DOES NOT NEED TO BE EDITED

search_endpoint = 'https://api.xbrl.us/api/v1/' + endpoint + '/search'
if unique:
    search_endpoint += '?unique'
orig_fields = params['fields']
res_df = []
query_start = datetime.now()

import math
total_keywords = len(Keyword_List)
keyword_batch_num = 1
rounds = math.ceil(total_keywords/keyword_batch_num)
round_num = 1
total_rows = 0
your_limit = 0

for x in range(0, total_keywords, keyword_batch_num):
    offset_value = 0
    count = 0
    offset_value = 0
    printed = False
    run_query = True
    segment_query_start = datetime.now()
    params['label.text'] = ','.join(Keyword_List[x:x+keyword_batch_num])
    params['fields'] = orig_fields
    print('Round %d/%d keyword "%s"' % (round_num, rounds, ','.join(Keyword_List[x:x+keyword_batch_num])))
    res_df_segment = []
    #print(params)

    while True:
        if not printed:
            print('On', query_start.strftime('%c'), tokenInfo.email, '(client ID:', str(tokenInfo.client_id.split('-')[0]), '...) started the query ')
            printed = True
        retry = 0
        while retry < 3:
            res = requests.get(search_endpoint, params=params, headers={'Authorization' : 'Bearer {}'.format(tokenInfo.access_token)})
            res_json = res.json()
            if 'error' in res_json:
                if res_json['error_description'] == 'Bad or expired token':
                    tokenInfo = refresh(tokenInfo)
                else:
                    print('There was an error: {}'.format(res_json['error_description']))
                    run_query = False
                    break
            else:
                    break
            retry +=1
            if retry >= 3:
                print('Cannot refresh the access token.  Run the first query block, then rerun the query.')
                run_query = False

        if not run_query:
            break

        print('up to', str(offset_value + res_json['paging']['limit']), 'records are found so far ...')

        res_df_segment += res_json['data']
        your_limit = res_json['paging']['limit']

        if res_json['paging']['count'] < res_json['paging']['limit']:
            print(' - this set contained fewer than the', res_json['paging']['limit'], 'possible, only', str(res_json['paging']['count']), 'records.')
            break
        else:
            offset_value += res_json['paging']['limit']
            if 100 == res_json['paging']['limit']:
                    params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                    if offset_value == 10 * res_json['paging']['limit']:
                            break
            elif 500 == res_json['paging']['limit']:
                    params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                    if offset_value == 4 * res_json['paging']['limit']:
                            break
            params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)

    current_datetime = datetime.now().replace(microsecond=0)
    time_taken = current_datetime - segment_query_start
    print('\nAt %s the query %s finished with  %d rows returned in %s.\n%s\n' % (current_datetime.strftime("%c"), params['label.text'], len(res_df_segment), str(time_taken), urllib.parse.unquote(res.url)))
    total_rows += len(res_df_segment)
    round_num += 1
    res_df += res_df_segment
    your_limit = res_json['paging']['limit']
    limit_message = 'If the results below match the limit noted above, you might not be seeing all rows, and should consider upgrading (https://xbrl.us/access-token).\n'

    if your_limit == 100:
        print('\nThis non-Member account has a limit of ' , 10 * your_limit, ' rows per query from our Public Filings Database. ' + limit_message)
    elif your_limit == 500:
        print('\nThis Basic Individual Member account has a limit of ', 4 * your_limit, ' rows per query from our Public Filings Database. ' + limit_message)

if not 'error' in res_json:
    current_datetime = datetime.now().replace(microsecond=0)
    all_time_taken = current_datetime - query_start
    index = pd.DataFrame(res_df).index
    all_rows = len(index)
    
    df = pd.DataFrame(res_df)

extension_df = df.copy()
extension_rows = 0
unique_label = df['label.text'].nunique()
unique_concept = df['concept.local-name'].nunique()
all_dts_concept = df.groupby(['concept.local-name', 'dts.id']).nunique()
ext_concept = df[~df['concept.namespace'].str.contains(r'://fasb\.org|://sec\.gov', na=False, regex=True)]
ext_dts_concept = ext_concept.groupby(['concept.local-name', 'dts.id'])
report_count = ext_concept['concept.namespace'].nunique()
total_count = df.shape[0]

if total_count > 0:
  base_percentage = (len(ext_dts_concept) / len(all_dts_concept)) * 100 
  print(f'\nThe dataframe of {all_rows} rows was created in {str(all_time_taken)} at {current_datetime.strftime("%Y-%m-%d %H:%M:%S")}. \nIt contains {unique_label} labels for {unique_concept} unique concepts throughout {report_count} reports (any non-SEC reports will be excluded during fact querying). \nOf the {len(all_dts_concept)} unique concept and dts combinations, {len(ext_dts_concept)} are extension concepts ({int(base_percentage)}%).\n')
else:
  print('There are no reports.')
df.head(10)

# Filter for extension concepts (optional)

Run the next cell to **exclude US GAAP and SEC taxonomy concepts**. In these cases, the entity applied a preferred label to a US GAAP or SEC taxonomy concept to help the reader understand the context for the fact while keeping the data within the base taxonomy.

To use all concepts from the `label` dataframe - including US GAAP and SEC taxonomy concepts - _skip this cell_.

In [ ]:
extensions = df[~df['concept.namespace'].str.contains(r'://fasb\.org|://sec\.gov', na=False, regex=True)].drop_duplicates(subset=['concept.local-name', 'dts.id'])
extension_df = extensions[['concept.local-name', 'concept.id', 'dts.id', 'concept.namespace']]
extension_rows = len(extension_df)

base_df = df[df['concept.namespace'].str.contains(r'://fasb\.org|://sec\.gov', na=False, regex=True)].drop_duplicates(subset=['concept.local-name', 'dts.id'])
base_df = base_df[['concept.local-name', 'concept.id', 'dts.id', 'concept.namespace']].sort_values(by=['concept.local-name', 'concept.namespace', ], ascending=[True, False])
base_rows = len(base_df)
print(f'The fact query loop below will use the dataframe of {str(extension_rows)} extension concepts matching keyword strings. \n{base_rows} base concepts in the dataframe will not be used for the evaluation. \nTo use all concepts, re-run the parameter and query cells above and skip this cell.')

extension_df.head(10)
#base_df.head(10)

The next cell can be run to save the full (or filtered) `label` dataframe as .csv

In [ ]:
# If you run this program locally, you can save the output to a file
# on your computer (modify D:\label-results.csv to your system)

extension_df.to_csv(r'D:\label-results.csv',sep=',')

# Google Colab users - comment out the line above and uncomment the code below to save the data frame as a .csv in your Google Drive

#from google.colab import drive
#drive.mount('drive')
#extension_df.to_csv('label-results.csv')
#!cp label-results.csv 'drive/My Drive/'

# Get facts for the concepts in the dataframe

Edit parameters for the fact query below and run it to input the number of keyword dataframe label rows and/or the starting row in the dataframe to process for facts (useful for evaluating a small sample of fact results or targeting a specific `dts.id` range). Then run the next cell to iterate `concept.local-name` and `dts.id` from the dataframe above in a fact query and create a de-duplicated dataframe of facts matching the concepts from the label keyword query in reports for defined fiscal periods and years.

In [ ]:
results_to_display = int(20) # Set as int() to display all rows
period_year = [2020, 2021, 2022, 2023, 2024,]
period = ['Y']
fields = [
         'concept.is-base',
         'concept.id',
         'entity.name',
         'dimensions.count',
         'concept.local-name',
         'fact.value',
         'period.fiscal-period',
         'period.fiscal-year',
         'period.instant',
         'dimension-pair',
         'dts.id',
         'report.filing-date',
         'report.document-type',
         'report.sic-code',
         'report.sec-url'
         ]

unique_df = pd.DataFrame(extension_df[['dts.id', 'concept.local-name']]).drop_duplicates() # Create a unique list of concept and dts details from the label query to iterate as facts
unique_rows = len(unique_df)
index_rows =  unique_rows
start_row = int(0)
index_rows = int(input(f'Enter a number between 1 and {unique_rows} of unique reported concepts from the label dataframe to query for the latest {"".join(period)} facts. \nThe dataframe of unique facts will be filtered for fiscal years {", ".join(map(str, period_year))}. Leave the field blank to build all facts: ') or index_rows)
start_row = int(input(f'\n\nEnter a starting row number between {start_row} and {unique_rows} to evaluate concepts or leave blank to start from the first row:') or start_row)
print('\n\nClick the run button below to execute this query for ', index_rows, ' qualifying concepts starting from row ', start_row, '\n\n')
unique_df_start = unique_df.iloc[start_row:]

In [ ]:
# @title
import time
row_count = 0
fact_query_start = datetime.now().replace(microsecond=0)
print('Starting from row ', start_row, ' of the keyword dataframe and querying  ', index_rows, 'extension' if extension_rows != 0 else 'unfiltered', '  concepts for the latest ', str("".join(period)), ' facts started at ', fact_query_start)
facts = []
fact_final = []
fact_res_df_segment = []
print_percent = set()
for y, row in unique_df.iloc[start_row:].iterrows():
    fact_search_endpoint = 'https://api.xbrl.us/api/v1/fact/search'
    fact_res_df = []
    fact_params = {
        'concept.local-name': str(row['concept.local-name']),
        'dts.id': str(row['dts.id']),
        'report.source-name': 'SEC',
        'period.fiscal-period': 'Y',
        'concept.is-monetary': 'TRUE',
        'fact.accuracy-index': '1',
        'fact.ultimus' : 'TRUE',
        'fields': ','.join(fields)
    }
    retry = 0  # Initialize retry counter within the loop
    run_query = True  # Initialize run_query within the loop
    #print(row['dts.id'])
    if row_count <= index_rows:
        while True:
            fact_res = requests.get(fact_search_endpoint, params=fact_params, headers={'Authorization': 'Bearer {}'.format(tokenInfo.access_token)})
            fact_res_json = fact_res.json()
            #time.sleep(1)
            #print(urllib.parse.unquote(fact_res.url))
            if 'error' in fact_res_json:
                if fact_res_json['error_description'] == 'Bad or expired token':
                    tokenInfo = refresh(tokenInfo)
                else:
                    print(f"Error for concept.local-name: {str(row['concept.local-name'])}, dts.id: {str(row['dts.id'])}: {fact_res_json['error_description']}")  # Print error with context
                    run_query = False
                    break  # Exit retry loop if there's an error other than token expiry
            else:
                break  # Exit retry loop if successful
            retry += 1
            if retry >= 3:
                print('Cannot refresh the access token.  Run the first query block, then rerun the query.')
                break  # Exit the main loop if token refresh fails repeatedly

        if run_query and 'data' in fact_res_json:  # Check if query was successful and 'data' key exists
            #print(fact_res_json['data'])
            fact_res_df_segment.extend(fact_res_json['data'])
            total_facts = len(fact_res_df_segment)
            percent_complete = int(row_count/index_rows * 100)
            if percent_complete in (1, 5, 10, 20, 40, 50, 60, 80, 90) and percent_complete not in print_percent:
                percent_datetime = datetime.now().replace(microsecond=0)
                percent_time = percent_datetime - fact_query_start
                print(f'With {percent_complete}% complete in {percent_time} ({row_count} keyword dataframe concepts evaluated), {total_facts} facts are in the dataframe.')
            print_percent.add(percent_complete)
        row_count += 1

if not 'error' in fact_res_json:
    current_datetime = datetime.now().replace(microsecond=0)
    time_taken = current_datetime - fact_query_start
    facts = pd.DataFrame(fact_res_df_segment)

for column in facts.columns:
    if facts[column].apply(lambda x: isinstance(x, list)).any():
        facts[column] = facts[column].astype(str)  # Convert to strings
    
filtered_facts = facts[facts['period.fiscal-year'].isin(period_year)]
fact_final = pd.DataFrame(filtered_facts.drop_duplicates())
fact_final.sort_values(by=['dts.id', 'concept.is-base', 'concept.local-name', 'dimensions.count','period.fiscal-year'], ascending=[False, False, True, True, False], inplace=True)
final_count = len(fact_final)
top_10 = fact_final['concept.local-name'].value_counts().head(10)
dimensions_fact = fact_final[fact_final['dimensions.count'] != 0]
dimensions_count = dimensions_fact.shape[0]
dimensions_summary = dimensions_fact['dimensions.count'].value_counts()

print('\nThe iteration of ', index_rows, ' rows starting from, ', start_row, ' took ', time_taken, ' and created a dataframe of ', total_facts, ' unique reported facts. \nOf these, ' ,final_count, ' occurred in the ',str("".join(period)), ' period for years', ', '.join(map(str, period_year)), 'across entities using label text that matched keyword strings for reported concepts (NOTE: labels can change across reporting periods). \nThese are the top 10 concepts in the dataframe: \n',top_10, '\n\nHere is a summary of the number of dimensionalized facts in the dataframe: \nTotal: ', dimensions_count, '\n',dimensions_summary, ' \n\nAn excerpt of ', results_to_display, ' rows below was produced by queries similar to ',urllib.parse.unquote(fact_res.url),'\n\n')
fact_final.head(results_to_display)

This cell can be run to save the `fact_final` dataframe of reported facts within the defined periods for fiscal years as .csv

In [ ]:
# If you run this program locally, you can save the output to a file
# on your computer (modify D:\keyword-facts.csv to your system)

facts.to_csv(r'D:\keyword-facts.csv',sep=',')

# Google Colab users - comment out the line above and uncomment the code below to save the data frame as a .csv in your Google Drive

#from google.colab import drive
#drive.mount('drive')
#fact_final.to_csv('keyword-facts.csv')
#!cp keyword-facts.csv 'drive/My Drive/'